# Analytic Method: CS-01 TAS

**Purpose**: solve the Tele Assistance System as an open Jackson queueing network in closed form (M/M/c/K per node) across the full adaptation axis and check it against the Camara 2023 R1 / R2 / R3 targets.

**Inputs**: `data/config/profile/{dflt,opti}.json` (PACS-style Variable dicts) and `data/reference/baseline.json` (R1 / R2 / R3 thresholds).

**Outputs**:
- `data/results/analytic/<adaptation>/<profile>.json` - per-run metrics (nodes + network + routing + lambda_z).
- `data/results/analytic/<adaptation>/requirements.json` - R1 / R2 / R3 verdicts.
- `data/img/analytic/<adaptation>/*.png` - topology, heatmap, bars, delta figures.

**Equivalent CLI** (reproduces the four runs written by this notebook):
```bash
python -m src.methods.analytic --adaptation baseline
python -m src.methods.analytic --adaptation s1
python -m src.methods.analytic --adaptation s2
python -m src.methods.analytic --adaptation aggregate
```

This notebook is thin: all logic lives in `src.methods.analytic` and `src.view.qn_diagram`. The cells below just orchestrate, display, and save figures.

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd

from src.methods.analytic import run
from src.view import (
    plot_qn_topology,
    plot_qn_topology_grid,
    plot_nd_heatmap,
    plot_net_bars,
    plot_net_delta,
)

_IMG_ROOT = Path("data/img/analytic")
_ADAPTATIONS = ["baseline", "s1", "s2", "aggregate"]

## 1. Solve every adaptation

`run(adp=a, wrt=True)` loads the resolved `NetworkConfig`, solves the Jackson network, writes the per-run envelope under `data/results/analytic/<adaptation>/`, and returns the per-node + aggregate DataFrames plus the R1 / R2 / R3 verdict.

In [ ]:
_results = {_a: run(adp=_a, wrt=True) for _a in _ADAPTATIONS}

# unpack the per-adaptation pieces used in every later cell
_cfgs = {_a: _results[_a]["config"] for _a in _ADAPTATIONS}
_nodes = {_a: _results[_a]["nodes"] for _a in _ADAPTATIONS}
_nets = {_a: _results[_a]["network"] for _a in _ADAPTATIONS}
_reqs = {_a: _results[_a]["requirements"] for _a in _ADAPTATIONS}

print(f"Solved {len(_results)} adaptations; wrote JSONs under data/results/analytic/")

## 2. Network-wide summary + verdict

One row per adaptation: headline metrics (response time, utilisation, queue length) and the R1 / R2 / R3 pass flags.

In [ ]:
_rows = []
for _a in _ADAPTATIONS:
    _n = _nets[_a].iloc[0]
    _r = _reqs[_a]
    _rows.append({
        "adaptation": _a,
        "profile": _cfgs[_a].profile,
        "W_net (ms)": _n["W_net"] * 1000,
        "avg_rho": _n["avg_rho"],
        "max_rho": _n["max_rho"],
        "L_net": _n["L_net"],
        "R1": "PASS" if _r["R1"]["pass"] else "FAIL",
        "R2": "PASS" if _r["R2"]["pass"] else "FAIL",
        "R3": "PASS" if _r["R3"]["pass"] else "FAIL",
    })
_summary = pd.DataFrame(_rows).set_index("adaptation")
_summary.round(4)

## 3. Per-node snapshot (baseline)

The reference configuration before any adaptation kicks in. `rho = lambda / (c * mu)`; `W`, `Wq` are in seconds; `L`, `Lq` are mean request counts.

In [ ]:
_nodes["baseline"][[
    "key", "name", "type", "lambda", "mu", "c", "K",
    "rho", "L", "W",
]].round(4)

## 4. Queue-network topology (architecture view)

Nodes are coloured by `rho` (cool = low, warm = high). Edge labels show routing probabilities.

In [ ]:
plot_qn_topology(
    rout=_cfgs["baseline"].routing,
    nds=_nodes["baseline"],
    net=_nets["baseline"],
    title="Baseline Topology (Before Adaptation)",
    file_path=str(_IMG_ROOT / "baseline"),
    fname="topology.png")

In [ ]:
# side-by-side topology: baseline (before) vs each adaptation (after)
for _a in ["s1", "s2", "aggregate"]:
    plot_qn_topology_grid(
        routs=[_cfgs["baseline"].routing, _cfgs[_a].routing],
        ndss=[_nodes["baseline"], _nodes[_a]],
        names=["Baseline", _a],
        nets=[_nets["baseline"], _nets[_a]],
        title=f"Topology: Before vs After ({_a})",
        file_path=str(_IMG_ROOT / _a),
        fname="topology_vs_baseline.png")

## 5. Per-node heatmap (before vs after)

Each row = one artifact; each column = one metric. Columns are normalised per-metric across both scenarios so the heat value is directly comparable. Numeric cell labels show the raw value.

In [ ]:
_node_keys = _nodes["baseline"]["key"].tolist()
_heat_metrics = ["rho", "L", "Lq", "W", "Wq"]

for _a in ["s1", "s2", "aggregate"]:
    plot_nd_heatmap(
        ndss=[_nodes["baseline"], _nodes[_a]],
        names=["Baseline", _a],
        nodes=_node_keys,
        metrics=_heat_metrics,
        title=f"Per-node Metrics: Before vs After ({_a})",
        file_path=str(_IMG_ROOT / _a),
        fname="heatmap_vs_baseline.png")

## 6. Network-wide bars (all four adaptations)

Headline comparison of the four configurations on the metrics that drive the R1 / R2 / R3 verdicts. Y-axis is log-scaled because the metrics span several orders of magnitude (W in seconds vs L in requests).

In [ ]:
_bar_metrics = ["W_net", "avg_rho", "max_rho", "L_net"]

plot_net_bars(
    nets=[_nets[_a] for _a in _ADAPTATIONS],
    names=_ADAPTATIONS,
    metrics=_bar_metrics,
    title="Network Metrics: All Adaptations",
    file_path=str(_IMG_ROOT / "aggregate"),
    fname="net_bars_all.png")

## 7. Network-wide delta (% change vs baseline)

For every non-baseline adaptation: fractional change on the headline metrics. Green bars = improvement, red = degradation. `total_throughput` flips the sign convention (more is better).

In [ ]:
_delta_metrics = ["W_net", "avg_rho", "max_rho", "L_net", "total_throughput"]
_bl = _nets["baseline"].iloc[0]

for _a in ["s1", "s2", "aggregate"]:
    _ac = _nets[_a].iloc[0]
    _row = {
        _m: (_ac[_m] - _bl[_m]) / _bl[_m] if _bl[_m] else 0.0
        for _m in _delta_metrics
    }
    plot_net_delta(
        deltas=pd.DataFrame([_row]),
        title=f"Network-wide Delta: {_a} vs Baseline",
        file_path=str(_IMG_ROOT / _a),
        fname="net_delta_vs_baseline.png")

## 8. R1 / R2 / R3 verdict table

Thresholds come from [`data/reference/baseline.json`](data/reference/baseline.json):

- **R1** Availability: `fail_rate <= 0.03 %` (fraction `0.0003`)
- **R2** Performance: `resp_time <= 26 ms`
- **R3** Minimise `cost` subject to `R1 and R2`

`R3` has no numeric threshold; it passes whenever both `R1` and `R2` do.

In [ ]:
_req_rows = []
for _a in _ADAPTATIONS:
    _r = _reqs[_a]
    _req_rows.append({
        "adaptation": _a,
        "R1 fail_rate": _r["R1"]["value"],
        "R1 pass": _r["R1"]["pass"],
        "R2 resp_time (s)": _r["R2"]["value"],
        "R2 pass": _r["R2"]["pass"],
        "R3 pass": _r["R3"]["pass"],
    })
pd.DataFrame(_req_rows).set_index("adaptation")

## Summary

At the nominal 345 req/s arrival rate, all four adaptations pass R1, R2, and R3. The aggregate configuration (opti routing + opti services) delivers the lowest `W_net` and the lowest `max_rho`; s1 alone (opti routing, dflt services at the swap slots) is the worst on `max_rho` because opti routing pushes more load into dflt services.

**Next method in the pipeline**: `stochastic.ipynb` (SimPy DES) produces the ground-truth distribution this closed-form solution is compared against.